# Reranker Evaluation

Notebook for explicit evaluation of the cross-encoder reranker on top of the first-stage retriever.


## Pipeline Overview

Stages:
1. setup runtime and imports
2. load and preprocess data
3. prepare first-stage retriever
4. predict categories
5. build the cross-encoder reranker
6. run base retrieval on train queries
7. rerank those results
8. compare metrics and inspect bad cases


In [ ]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

import pandas as pd

from src.infra.notebook import setup_notebook
runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')


## Step 1: Load Data

Load the normalized train/test views and the training ground-truth labels.


In [ ]:
from src.config import DEFAULT_CONFIG
from src.evaluation import leaderboard_score, load_ground_truth, mrr_at_k
from src.output.diagnostics import build_bad_case_entry, build_scalar_map
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    rerank_retrieval_results,
    run_first_stage_retrieval,
)
from src.reranking import rerank_results_with_cross_encoder
from src.retrieval import truncate_results

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')


## Step 2: Prepare First-Stage Retrieval and Categories

Prepare the configured retriever and the category predictions needed for reranking and category-aware analysis.


In [ ]:
prepared_retrievers = prepare_retrievers(frames, paths, config=config)
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
print(f'Category accuracy: {category_artifacts.classifier_accuracy:.5f}')


## Step 3: Build Cross-Encoder

Load or train the cross-encoder reranker using the training labels.


In [ ]:
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)
if cross_encoder_reranker is None:
    raise ValueError('Cross-encoder reranking is disabled in config.')
print('Cross-encoder reranker is ready.')


## Step 4: Base Retrieval and Reranking

Run the first-stage retriever on the training queries, then apply cross-encoder reranking to the same candidate lists.


In [ ]:
eval_top_k = max(config.retrieval_pipeline.evaluation_top_ks)
base_results, _ = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split='train',
    top_k=eval_top_k,
    config=config,
)
reranked_results, rerank_diagnostics = rerank_results_with_cross_encoder(
    results=base_results,
    query_frame=frames.train_queries,
    docs_frame=frames.docs,
    cross_encoder=cross_encoder_reranker,
    query_category_map=category_artifacts.train_query_category_map,
    doc_category_map=category_artifacts.doc_category_map,
    infer_batch_size=config.cross_encoder.infer_batch_size,
    rerank_top_m=config.cross_encoder.rerank_top_m,
    category_bonus=config.cross_encoder.category_bonus if config.retrieval_pipeline.enable_category_filter else 0.0,
    return_diagnostics=True,
)


## Step 5: Compare Metrics

Measure how reranking changes Recall, Precision, MRR, Accuracy, and leaderboard score.


In [ ]:
metrics_rows = []
for top_k in config.retrieval_pipeline.evaluation_top_ks:
    base_metrics = leaderboard_score(truncate_results(base_results, top_k), ground_truth, k=top_k, accuracy_value=category_artifacts.classifier_accuracy)
    reranked_metrics = leaderboard_score(truncate_results(reranked_results, top_k), ground_truth, k=top_k, accuracy_value=category_artifacts.classifier_accuracy)
    metrics_rows.append({'Variant': 'base', 'TopK': int(top_k), **base_metrics})
    metrics_rows.append({'Variant': 'reranked', 'TopK': int(top_k), **reranked_metrics})

metrics_df = pd.DataFrame(metrics_rows)
metrics_df


In [ ]:
pivot = metrics_df.pivot(index='TopK', columns='Variant', values=['Recall', 'Precision', 'MRR', 'Accuracy', 'LeaderboardScore'])
pivot


## Step 6: Inspect Bad Cases

Build a small set of reranker regressions so failures are easy to inspect in the notebook.


In [ ]:
query_title_map = build_scalar_map(frames.train_queries_raw, 'title')
query_text_map = build_scalar_map(frames.train_queries_raw, 'text')
rerank_diag_by_query = {str(item['query_id']): item for item in rerank_diagnostics}
bad_case_candidates = []
for base_item, reranked_item in zip(base_results, reranked_results):
    query_id = str(base_item['query_id'])
    base_mrr = mrr_at_k([base_item], ground_truth, k=eval_top_k)
    reranked_mrr = mrr_at_k([reranked_item], ground_truth, k=eval_top_k)
    if reranked_mrr < base_mrr:
        bad_case_candidates.append({
            'query_id': query_id,
            'true_category': ground_truth[query_id].get('category'),
            'predicted_category': category_artifacts.train_query_category_map.get(query_id),
            'category_correct': ground_truth[query_id].get('category') == category_artifacts.train_query_category_map.get(query_id),
            'top_k': eval_top_k,
            'relevant_doc_ids': sorted(ground_truth[query_id]['relevant_doc_ids']),
            'retrieved_doc_ids': list(reranked_item['relevant_docs']),
            'base_retrieved_doc_ids': list(base_item['relevant_docs']),
            'hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in reranked_item['relevant_docs']),
            'base_hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in base_item['relevant_docs']),
            'first_relevant_rank': next((rank for rank, doc_id in enumerate(reranked_item['relevant_docs'], start=1) if doc_id in ground_truth[query_id]['relevant_doc_ids']), None),
            'base_first_relevant_rank': next((rank for rank, doc_id in enumerate(base_item['relevant_docs'], start=1) if doc_id in ground_truth[query_id]['relevant_doc_ids']), None),
            'delta_hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in reranked_item['relevant_docs']) - sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in base_item['relevant_docs']),
            'delta_recall_at_k': 0.0,
            'delta_precision_at_k': 0.0,
            'delta_reciprocal_rank': reranked_mrr - base_mrr,
            'rerank_diagnostics': rerank_diag_by_query.get(query_id),
        })

bad_cases = [
    build_bad_case_entry(item, query_title_map=query_title_map, query_text_map=query_text_map, doc_category_map=category_artifacts.doc_category_map)
    for item in sorted(bad_case_candidates, key=lambda row: row['delta_reciprocal_rank'])[:10]
]
bad_cases[:3]
